In [8]:
import sys
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("DeptSalaryMedianAssesment").getOrCreate()

df = spark.read.csv("/user/sclsepb456/payments_employee.csv", header=True, inferSchema=True)

median_df = df.groupBy("Department").agg(
    F.percentile_approx("Salary", 0.5).alias("median_salary")
)

joined_df = df.join(median_df, on="department", how="inner")

filtered_df = joined_df.filter(F.col("Salary") > F.col("median_salary"))

total_Count = filtered_df.count()

print(total_Count)

spark.stop()

50004


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, col, sum as _sum,current_timestamp

In [16]:
spark = SparkSession.builder \
    .appName("EmployeeSalaryStreaming") \
    .getOrCreate()

In [26]:
# Set shuffle partitions to a small number suitable for local/console testing
spark.conf.set("spark.sql.shuffle.partitions", "2")

df_check = spark.read.option("header", "true").csv("/user/sclsepb456/Raw/payment_employee/payments_employee.csv")
print(df_check.columns)
df_check.show(5, truncate=False)

['First Name', 'Last Name', 'Email', 'Phone', 'Gender', 'Age', 'Job Title', 'Years Of Experience', 'Salary', 'Department']
+----------+---------+-----------------------------+----------+------+---+---------------+-------------------+------+----------+
|First Name|Last Name|Email                        |Phone     |Gender|Age|Job Title      |Years Of Experience|Salary|Department|
+----------+---------+-----------------------------+----------+------+---+---------------+-------------------+------+----------+
|Pooja     |Singh    |pooja.singh11@email.com      |9000000011|Male  |32 |Data Analyst   |25                 |24574 |Analytics |
|Riya      |Malhotra |riya.malhotra99983@email.com |9000099983|Female|49 |HR Executive   |1                  |80048 |HR        |
|Neha      |Mehta    |neha.mehta99975@email.com    |9000099975|Male  |27 |Operations Lead|2                  |118660|Operations|
|Amit      |Patel    |amit.patel99933@email.com    |9000099933|Female|37 |Operations Lead|3            

In [28]:
# 2. Define Schema explicitly for the employee dataset
# Adjust field names based on your exact CSV headers (e.g., employee_id, name, department, salary, event_time)
employee_schema = StructType([
    StructField("First Name", StringType(), True),
    StructField("Last Name", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Phone", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Job Title", StringType(), True),
    StructField("Years Of Experience", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("event_time",TimestampType(),True)
])

raw_streaming_df = spark.readStream.schema(employee_schema).option("header","true").csv("/user/sclsepb456/Raw/payment_employee/")
streaming_with_time_df = raw_streaming_df.withColumn("event_time", current_timestamp())

aggregated_df = streaming_with_time_df.withWatermark("event_time", "5 minutes") \
    .groupBy(window(col("event_time"),"10 minutes"), col("department")) \
    .agg(_sum("salary").alias("total_salary")) \
    .orderBy(col("total_salary").desc())

query = aggregated_df.writeStream.outputMode("complete").format("console").option("truncate","false") \
    .start()

26/10/21 03:06:55 WARN StreamingQueryManager: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7d271985-5b87-4b48-ac59-a70c159f4738. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+----------+------------+
|window                                    |department|total_salary|
+------------------------------------------+----------+------------+
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|HR        |965179948   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|IT        |964920050   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|Product   |960371530   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|Finance   |957987416   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|Sales     |953286127   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|Operations|951268901   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|Analytics |938718182   |
|{2026-10-21 03:00:00, 2026-10-21 03:10:00}|Marketing |938475908   |
+------------------------------------------+----------+------------+



In [ ]:
query.awaitTermination()